# 2 · Train the U-Net surrogate (GPU runtime)

* **Runtime:** GPU (T4 or L4 is enough). Training resumes from `checkpoints/last.pt` if the session dies.
* The dataset is copied from Drive to the local disk first; reading thousands of small files from Drive is slow.

In [ ]:
# --- Project location on Google Drive -------------------------------------
# Mounts Drive only if it is not mounted yet. NEVER run `rm -rf /content/drive`:
# if Drive is mounted it deletes your whole Google Drive.
import os, subprocess
if not os.path.ismount('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/windsurrogate'      # change if you like
PROJECT    = f'{DRIVE_ROOT}/idaho'                        # one folder per dataset (regions in config)
CODE_ZIP   = f'{DRIVE_ROOT}/windsurrogate_code.zip'       # upload the code zip here once
assert os.path.ismount('/content/drive'), 'Drive is not mounted'
assert os.path.exists(CODE_ZIP), f'missing {CODE_ZIP}: upload the code zip there first'
os.makedirs(PROJECT, exist_ok=True)

# --- Code ---------------------------------------------------------------------
if not os.path.exists('/content/windsurrogate'):
    subprocess.run(['unzip', '-q', '-o', CODE_ZIP, '-d', '/content'], check=True)
%cd /content/windsurrogate
!pip -q install rasterio pyproj pyyaml

In [ ]:
!rsync -a --info=progress2 {PROJECT}/dataset/ /content/dataset/
cfg_text = f"""
project_dir: {PROJECT}
output:
  dataset_dir: /content/dataset
train:
  checkpoint_dir: {PROJECT}/checkpoints
  num_workers: 2
"""
open('/content/train.yaml', 'w').write(cfg_text)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!python -m windsurrogate --config /content/train.yaml train

In [ ]:
import json, pandas as pd
log = [json.loads(l) for l in open(f'{PROJECT}/checkpoints/train_log.jsonl')]
pd.DataFrame(log)[['epoch', 'train_loss', 'loss'] + [k for k in log[-1] if k.startswith('rmse_u') or k.startswith('recirc')]].tail(10)

## Evaluate on the test split
The test blocks were never used for training or for picking the best epoch, so these are the numbers to report.
Writes `metrics_test.json`, `cases_test.csv` and CFD-vs-U-Net figures to `checkpoints/eval_test/`.

In [ ]:
!python -m windsurrogate --config /content/train.yaml evaluate --checkpoint {PROJECT}/checkpoints/best.pt --split test --figures 8

In [ ]:
import glob
from IPython.display import Image, display
for f in sorted(glob.glob(f'{PROJECT}/checkpoints/eval_test/figures/*.png'))[:4]:
    display(Image(f, width=900))